In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict


def locate_file(filename: str) -> Path:
    """
    Search for *filename* in typical Kaggle notebook locations, including absolute
    paths used by Kaggle kernels. Returns the first matching Path.
    Raises FileNotFoundError if none found.
    """
    candidates = [
        Path("data") / filename,
        Path("data") / "stanford-covid-vaccine" / filename,
        Path("input") / filename,
        Path("input") / "stanford-covid-vaccine" / filename,
        Path("kaggle") / "data" / filename,
        Path("kaggle") / "data" / "stanford-covid-vaccine" / filename,
        Path("kaggle") / "input" / filename,
        Path("kaggle") / "input" / "stanford-covid-vaccine" / filename,
        Path("working") / "stanford-covid-vaccine" / filename,
        Path("working") / filename,
        Path("/kaggle/input") / filename,
        Path("/kaggle/input") / "stanford-covid-vaccine" / filename,
        Path("/kaggle/working") / filename,
        Path("/kaggle/working") / "stanford-covid-vaccine" / filename,
        Path(filename),
    ]
    for p in candidates:
        if p.is_file():
            return p
    raise FileNotFoundError(f"Unable to locate {filename} in any expected location.")


train_path = locate_file("train.json")
test_path = locate_file("test.json")

train_df = pd.read_json(train_path, lines=True)
test_df = pd.read_json(test_path, lines=True)




In [2]:
targets = ["reactivity", "deg_Mg_pH10", "deg_pH10", "deg_Mg_50C", "deg_50C"]
scored_len = int(train_df["seq_scored"].iloc[0])

overall_weighted_sum = {t: 0.0 for t in targets}
overall_weight_total = {t: 0.0 for t in targets}
pos_weighted_sum = {t: np.zeros(scored_len) for t in targets}
pos_weight_total = {t: np.zeros(scored_len) for t in targets}
nucleotide_weighted_sum = {t: defaultdict(lambda: defaultdict(float)) for t in targets}
nucleotide_weight_total = {t: defaultdict(lambda: defaultdict(float)) for t in targets}

for _, row in train_df.iterrows():
    seq = row["sequence"]
    weight = float(row.get("signal_to_noise", 1.0))
    for t in targets:
        vals = np.array(row[t])  # length = scored_len
        overall_weighted_sum[t] += vals.sum() * weight
        overall_weight_total[t] += vals.size * weight
        pos_weighted_sum[t] += vals * weight
        pos_weight_total[t] += weight
        for pos in range(scored_len):
            base = seq[pos]
            nucleotide_weighted_sum[t][pos][base] += vals[pos] * weight
            nucleotide_weight_total[t][pos][base] += weight

overall_mean = {}
pos_means = {}
nucleotide_means = {t: defaultdict(dict) for t in targets}

for t in targets:
    overall_mean[t] = (
        overall_weighted_sum[t] / overall_weight_total[t]
        if overall_weight_total[t] != 0
        else 0.0
    )
    pos_means[t] = np.divide(
        pos_weighted_sum[t],
        pos_weight_total[t],
        out=np.zeros_like(pos_weighted_sum[t]),
        where=pos_weight_total[t] != 0,
    )
    for pos in range(scored_len):
        for base, w_sum in nucleotide_weighted_sum[t][pos].items():
            w_total = nucleotide_weight_total[t][pos][base]
            if w_total > 0:
                nucleotide_means[t][pos][base] = w_sum / w_total
            else:
                nucleotide_means[t][pos][base] = None




In [3]:
# Smaller smoothing moves predictions toward the nucleotide‑position specific means,
# while a tiny overall‑mean blend regularises extremely rare cases.
SMOOTHING_BETA = 0.2  # less pull toward generic position mean
OVERALL_ALPHA = 0.1  # slight pull toward overall column mean

rows = []
for _, row in test_df.iterrows():
    sample_id = row["id"]
    seq_len = int(row["seq_length"])  # expected 107
    seq = row["sequence"]
    for pos in range(seq_len):
        id_seqpos = f"{sample_id}_{pos}"
        preds = []
        for t in targets:
            if pos < scored_len:
                base = seq[pos]
                nuc_mean = nucleotide_means[t][pos].get(base)
                nuc_weight = nucleotide_weight_total[t][pos].get(base, 0.0)
                if nuc_mean is not None and nuc_weight > 0:
                    # Blend nucleotide‑specific mean, position mean, and overall mean
                    pred = (
                        nuc_weight * nuc_mean
                        + SMOOTHING_BETA * pos_means[t][pos]
                        + OVERALL_ALPHA * overall_mean[t]
                    ) / (nuc_weight + SMOOTHING_BETA + OVERALL_ALPHA)
                else:
                    # No nucleotide info → fall back to position + overall blend
                    pred = (
                        SMOOTHING_BETA * pos_means[t][pos]
                        + OVERALL_ALPHA * overall_mean[t]
                    ) / (SMOOTHING_BETA + OVERALL_ALPHA)
            else:
                # Positions beyond the scored region: use overall mean
                pred = overall_mean[t]
            preds.append(float(pred))
        rows.append([id_seqpos] + preds)

submission_df = pd.DataFrame(rows, columns=["id_seqpos"] + targets)




In [4]:
submission_path = Path("submission.csv")
submission_df.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path} with shape {submission_df.shape}")




Submission written to submission.csv with shape (25680, 6)


In [5]:
sample_sub_path = locate_file("sample_submission.csv")
sample_sub = pd.read_csv(sample_sub_path, nrows=1)
expected_cols = list(sample_sub.columns)
assert list(submission_df.columns) == expected_cols, "Column order mismatch!"
print("Column order matches the sample submission.")

Column order matches the sample submission.
